In [1]:
from typing import Optional
from pathlib import Path
import json
import datetime
from urllib.parse import urlparse
import pandas as pd
from webdav4.fsspec import WebdavFileSystem

%load_ext autoreload
%autoreload 2


### Test retrieving data from webdav

In [ ]:
# Read only public link

fs = WebdavFileSystem("https://collab.psa.es/public.php/webdav", auth=("CcosLNJKw4Dbg9C", ""))

# list
display(fs.ls(".", detail=False))



['20250430_process_timeseries.mat',
 '20250527_process_timeseries.mat',
 '20250604_process_timeseries.mat',
 '20250606_process_timeseries.mat',
 '20250609_process_timeseries.mat',
 '20250613_process_timeseries.mat',
 '20250620_process_timeseries.mat',
 '20250623_process_timeseries.mat',
 '20250625_process.csv',
 '20250625_process_timeseries.mat',
 '20250701_process.csv',
 '20250701_process_timeseries.mat',
 '20250711_process_timeseries.mat',
 '20250714_process.csv',
 '20250714_process_timeseries.mat',
 '20250725_process.csv',
 '20250725_process_timeseries.csv',
 '20250725_process_timeseries.mat',
 '20250728_process_timeseries.csv',
 '20250728_process_timeseries.mat',
 'Convertidor.m']

In [ ]:
fs = WebdavFileSystem(
    "https://collab.psa.es/public.php/webdav", 
    auth=("WR6MxyJsnZWi9xH", ""),
    chunk_size=1024 * 1024 * 10,  # 10 MB
    timeout=20,  # seconds
)

# list
display(fs.ls(".", detail=False))

test_path: Path = Path("./test_folder")

# check a path exists and folder creation
print(f"{fs.exists(test_path)=}")
fs.mkdir(test_path)
print(f"{fs.exists(test_path)=}")

# reading files
display(
    pd.read_csv(fs.open('environment.csv'), index_col=0)
)

# copying directories from local to webdav
fs.put("test_fld/", "test_folder/", recursive=True)

# copying file
fs.put("/tmp/tmpg6rin0s8/optimization_results.tar.gz", "test_folder/test.tar.gz")

['20250725.7z',
 'environment.csv',
 'environment_at_20250728_1045.csv',
 'extended',
 'old',
 'optimization_results.csv',
 'test_folder']

fs.exists(test_path)=True
fs.exists(test_path)=True


,Tamb_C,HR_pct,precip_mm,G_Gh,G_Dh,G_Bn,G_Gk,Q_kW,Tv_C,mv_kgh,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_l,Vavail_m3,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
fecha,,,,,,,,,,,,,,,,,,,,
2025-07-28 12:00:00+00:00,37.2,28.0,0.0,855.0,99.0,916.7,635.1,150.0,45.0,225.6,0.029,13.454,0.0,0.013,235.967,0.236,341.68,168.180,0.342,0.168
2025-07-28 12:20:00+00:00,37.1,28.0,0.0,855.7,99.3,913.0,633.8,150.0,45.0,225.6,0.029,13.310,0.0,0.013,235.967,0.236,338.79,166.373,0.339,0.166


In [ ]:
def get_data(data_url: str) -> pd.DataFrame:
    """
    Function to download data from a public URL and return it as a pandas DataFrame.
    """
    # Read the CSV file from the URL
    df = pd.read_csv(data_url)
    return df

def extract_url_components(url: str) -> tuple[str, str]:
    
    domain = urlparse(url).netloc
    share_id = urlparse(url).path.split('/')[-1]
    
    return domain, share_id

def build_file_url(
    file_id: str, 
    ext: str, 
    url: Optional[str] = None, 
    domain: Optional[str] = None, 
    share_id: Optional[str] = None
) -> str:
    """Builds a URL for accessing a file on a webdav server.
    If a full URL is provided, it extracts the domain and share_id from it.
    If only domain and share_id are provided, it constructs the URL accordingly.
    If share_id is None, it assumes the file is being uploaded to webdav.
    """
    
    assert (url is not None) or (domain is not None), \
        "Either a full URL or both domain and share_id must be provided."
    if url is not None:
        domain, share_id = extract_url_components(url)
        
    if share_id is None: # When uploading to webdav
        return f"https://{domain}/public.php/webdav/{file_id}.{ext}"
    else: # When downloading from webdav
        return f"https://{domain}/public.php/dav/files/{share_id}/{file_id}.{ext}"


url: str = "https://collab.psa.es/s/WR6MxyJsnZWi9xH"
env_file_id: str = "environment"

request_url = build_file_url(            
    url=url,
    file_id=env_file_id,
    ext="csv"
)
df = get_data(request_url)
df


In [6]:
df.iloc[0]["Unnamed: 0"]


'2022-01-01 00:00:00+00:00'

### Test retrieving previous results and updating to an specific date

In [ ]:
from pathlib import Path
import pandas as pd
import datetime
from solhycool_optimization import DayResults

def create_mock_day_results(date_str: str, template_path: Path, template_date_str: str = "20220501") -> DayResults:
    """
    Create a new DayResults object with the same content as the template,
    but with timestamps shifted to match the given date_str.
    """
    # 1. Load from existing file
    original = DayResults.initialize(template_path, date_str=template_date_str)

    # 2. Convert new date_str to datetime
    new_date = pd.to_datetime(date_str, format="%Y%m%d")
    old_date = pd.to_datetime(template_date_str, format="%Y%m%d")
    delta_days = (new_date - old_date).days

    # 3. Shift index and time-dependent fields
    new_index = original.index + pd.Timedelta(days=delta_days)
    new_df_results = original.df_results.copy()
    new_df_results.index = new_df_results.index + pd.Timedelta(days=delta_days)

    # Optional: shift fitness history index if needed (not always datetime-indexed)
    fitness_history = original.fitness_history.copy()
    if isinstance(fitness_history.index, pd.DatetimeIndex):
        fitness_history.index = fitness_history.index + pd.Timedelta(days=delta_days)

    return DayResults(
        index=new_index,
        df_paretos=original.df_paretos,  # usually per-timestep, no timestamp inside
        fitness_history=fitness_history,
        selected_pareto_idxs=original.selected_pareto_idxs,
        df_results=new_df_results,
        consumption_arrays=original.consumption_arrays,
        pareto_idxs=original.pareto_idxs,
        date_str=date_str
    )

# Use current date
mock_date_str = datetime.datetime.today().strftime("%Y%m%d")

mock_day_results = create_mock_day_results(
    date_str=mock_date_str,
    template_path=Path("../dags/results_eval_at_20250421T1741_psa_partial.gz"),
    template_date_str="20220501"
)

# Now you can export or use mock_day_results downstream
mock_day_results.export(Path(f"/tmp/mock_day_results_{mock_date_str}.h5"))


2025-07-21 08:13:30.873 | INFO     | solhycool_optimization:initialize:201 - DayResults loaded for 20220501 from ../dags/results_eval_at_20250421T1741_psa_partial.gz
2025-07-21 08:13:31.402 | INFO     | solhycool_optimization:export:267 - Results for 20250721 saved to /tmp/mock_day_results_20250721.h5


In [6]:
mock_day_results.index[0].dt


AttributeError: 'Timestamp' object has no attribute 'dt'

### Test generating visualizations for the results

In [17]:
len(day_results.index)


30

In [ ]:
len(day_results.)


30

In [2]:
from pathlib import Path
import hjson
from solhycool_optimization import DayResults
from solhycool_visualization.objects import HorizonResultsVisualizer

plot_config = hjson.loads(Path("../../data/plot_config_day_horizon.hjson").read_text())
day_results2 = DayResults.initialize(Path("./multi_day_results.h5"), date_str=None)

visualizer = HorizonResultsVisualizer(
    results_plot_config=plot_config,
    day_results=day_results2,
)

[display(fig) for fig in visualizer.plot_pareto_fronts()]

visualizer.plot_results()
# visualizer.generate_all(
#     output_path=Path("."),
#     formats=["png"]
# )


2025-07-23 07:38:02.500 | INFO     | solhycool_optimization:initialize:228 - DayResults loaded for 20240523T2000-20240524T0500 from multi_day_results.h5


### Test evaluating optimization and multi-day results

In [1]:
# read_environment task
from deployment import get_data, extract_url_components, build_file_url

url = "https://collab.psa.es/s/WR6MxyJsnZWi9xH"
file_id = "environment"

request_url = build_file_url(            
    url=url,
    file_id=file_id,
    ext="csv"
)

df_env = get_data(request_url)
# df_env.iloc[20:30]


In [2]:
# evaluate_optimization task
from dataclasses import asdict
import numpy as np
import pandas as pd
from solhycool_optimization import ValuesDecisionVariables
from solhycool_optimization.problems.horizon.evaluation import evaluate_day
from solhycool_optimization.problems.horizon import AlgoParams

n_parallel_steps: int = 10
values_per_decision_variable: int = 5
algo_params: AlgoParams = AlgoParams()

df_env.index = pd.to_datetime(df_env.index)

# To not spend all day trying
# df_env = df_env.iloc[20:30]  # Use only the first day for testing

# date_str = df_env.index[0].strftime("%Y%m%d")
dv_values=ValuesDecisionVariables.initialize(
    values_per_dv=values_per_decision_variable
).generate_arrays()

# print(dv_values)

# Compute total number of evaluations
total_num_evals = np.prod([len(value) for value in asdict(dv_values).values()])

day_results = evaluate_day(
    n_parallel_evals=n_parallel_steps,
    df_day=df_env, 
    dv_values=dv_values, 
    total_num_evals=total_num_evals, 
    path_selector_params=algo_params,
)

day_results.index


2025-07-23 09:13:30.762 | INFO     | solhycool_optimization:initialize:88 - Initializing ValuesDecisionVariables with 5 values per decision variable. A total of 625 combinations will be generated.
Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Error using evaluate_operation (line 27)
Invalid argument at position 3. Value must be positive.

Error using evaluate_operation (line 27)
Invalid argume

MatlabRuntimeError: An error occurred when evaluating the result from a function. Details: 
  File /root/.MathWorks/MatlabRuntimeCache/R2024b/combin8/combined_coo/evaluate_operation.m, line 27, in evaluate_operation
Invalid argument at position 3. Value must be positive.


In [4]:
from pathlib import Path

day_results.export(output_path=Path("./multi_day_results.h5"), single_day=False, overwrite=True)


2025-07-23 07:26:25.326 | INFO     | solhycool_optimization:export:302 - Results for 20240523 saved to multi_day_results.h5


In [13]:
df.Cw_s2


time
2024-05-23 20:00:00+00:00      0.000000
2024-05-23 21:00:00+00:00    102.731957
2024-05-23 22:00:00+00:00    131.206021
2024-05-23 23:00:00+00:00    132.768806
2024-05-24 00:00:00+00:00    131.556985
2024-05-24 01:00:00+00:00    131.446476
2024-05-24 02:00:00+00:00    130.054005
2024-05-24 03:00:00+00:00    132.216149
2024-05-24 04:00:00+00:00    135.299350
2024-05-24 05:00:00+00:00     26.610219
Name: Cw_s2, dtype: float64

In [7]:
from pathlib import Path
from solhycool_optimization import DayResults, import_simulation_results

# template_path = Path("/workspaces/SOLhycool/deployment/dags/day_results.h5")

df = import_simulation_results(Path("./multi_day_results.h5"))
display(df)

day_results2 = DayResults.initialize(Path("./multi_day_results.h5"), date_str=None)
day_results2.index


,Tamb,HR,mv,Tv,qc,Tc_in,Tc_out,Tcond,Qc_released,Qc_absorbed,...,Je_c,Je_dc,Je_wct,Jw_wct,Jw_s1,Jw_s2,dc_active,dc_mode,wct_active,wct_mode
time,,,,,,,,,,,,,,,,,,,,,
2024-05-23 20:00:00+00:00,21.5,45.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.251743,131.765502,48.741182,0.003558,0.003558,0.000000,True,0,True,0
2024-05-23 21:00:00+00:00,20.4,53.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.233606,122.272105,34.821776,0.704478,0.000558,0.703919,True,0,True,0
2024-05-23 22:00:00+00:00,19.3,60.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.231833,28.656581,53.143438,0.892201,0.000000,0.892201,True,0,True,0
2024-05-23 23:00:00+00:00,18.2,63.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.213668,26.411254,36.928667,0.832089,0.000000,0.832089,True,0,True,0
2024-05-24 00:00:00+00:00,17.7,61.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.211405,26.131431,33.806655,0.815759,0.000000,0.815759,True,0,True,0
2024-05-24 01:00:00+00:00,17.0,62.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.196376,24.273810,28.089546,0.757132,0.000000,0.757132,True,0,True,0
2024-05-24 02:00:00+00:00,16.7,60.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.193649,23.936674,26.805345,0.738707,0.000000,0.738707,True,0,True,0
2024-05-24 03:00:00+00:00,16.3,65.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.221169,27.338379,28.554041,0.857713,0.000000,0.857713,True,0,True,0
2024-05-24 04:00:00+00:00,16.0,72.0,297.774165,35.0,19.4210,24.564011,33.435989,35.0,200.0,200.065193,...,0.227306,28.096935,28.202745,0.902068,0.000000,0.902068,True,0,True,0


2025-07-22 18:53:43.121 | INFO     | solhycool_optimization:initialize:228 - DayResults loaded for 20240523T2000-20240524T0500 from multi_day_results.h5


DatetimeIndex(['2024-05-23 20:00:00+00:00', '2024-05-23 21:00:00+00:00',
               '2024-05-23 22:00:00+00:00', '2024-05-23 23:00:00+00:00',
               '2024-05-24 00:00:00+00:00', '2024-05-24 01:00:00+00:00',
               '2024-05-24 02:00:00+00:00', '2024-05-24 03:00:00+00:00',
               '2024-05-24 04:00:00+00:00', '2024-05-24 05:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='time', freq=None)